# Salesforce NPSP Fundraising Q&A with LlamaIndex

This notebook is an end-to-end walkthrough of the `SalesforceNPSPReader` integration.
It shows how to turn your Salesforce NPSP donor data into an AI-powered Q&A system using LlamaIndex.

**What you will learn:**
- How to initialize `SalesforceNPSPReader` with all available options
- How to load donor documents using SOQL filters or explicit Contact IDs
- How the returned `Document` objects are structured (text + metadata)
- How to attach a custom affinity score to every document at load time
- How to build a `VectorStoreIndex` and run natural-language fundraising queries
- How to pre-filter documents before indexing to focus on specific donor segments

**Prerequisites:**
- A Salesforce org with Nonprofit Success Pack (NPSP) installed
- A Salesforce user with API access and a security token
- An LLM + embedding model configured for LlamaIndex (OpenAI or non-OpenAI)
- Environment variables set: `SF_USERNAME`, `SF_PASSWORD`, `SF_TOKEN` (and `OPENAI_API_KEY` if using OpenAI)

## Install dependencies

The reader depends on `simple-salesforce` for Salesforce API access and `llama-index-core`
for the document model and indexing primitives.

In [ ]:
%pip install llama-index-readers-salesforce-npsp llama-index-core llama-index-llms-openai llama-index-embeddings-openai --quiet

## Step 1 — Verify credentials

`SalesforceNPSPReader` reads credentials from the `SF_USERNAME`, `SF_PASSWORD`, and `SF_TOKEN`
environment variables (or you can pass them directly as constructor arguments — see Step 2).
Run this cell first to confirm they are present before attempting a connection.

In [ ]:
import os

required = {"SF_USERNAME": "Salesforce username",
            "SF_PASSWORD": "Salesforce password",
            "SF_TOKEN":    "Salesforce security token"}

missing = [f"{var} ({desc})" for var, desc in required.items() if not os.getenv(var)]
if missing:
    raise EnvironmentError(
        "Missing required environment variables:\n  " + "\n  ".join(missing) + "\n"
        "Set them before running this notebook, e.g.:\n"
        "  export SF_USERNAME='user@example.com'\n"
        "  export SF_PASSWORD='yourpassword'\n"
        "  export SF_TOKEN='yoursecuritytoken'"
    )

print("All Salesforce credentials are present.")
print(f"  SF_USERNAME : {os.getenv('SF_USERNAME')}")
print(f"  SF_PASSWORD : {'*' * len(os.getenv('SF_PASSWORD', ''))}")
print(f"  SF_TOKEN    : {'*' * len(os.getenv('SF_TOKEN', ''))}")

## Step 2 — Initialize the reader

`SalesforceNPSPReader.__init__` accepts the following parameters:

| Parameter | Type | Default | Description |
|---|---|---|---|
| `username` | `str \| None` | `None` | Salesforce username. Falls back to `SF_USERNAME` env var. |
| `password` | `str \| None` | `None` | Salesforce password. Falls back to `SF_PASSWORD` env var. |
| `security_token` | `str \| None` | `None` | Salesforce security token. Falls back to `SF_TOKEN` env var. |
| `domain` | `str` | `"login"` | `"login"` for production orgs, `"test"` for sandboxes. |
| `include_opportunities` | `bool` | `True` | When `True`, gift history (Opportunity records) is fetched and appended to each document. |
| `affinity_score_fn` | `Callable[[dict], float] \| None` | `None` | Optional function that receives the metadata dict and returns a float score stored as `affinity_score`. |

The Salesforce connection is lazy — it is created only when the first `load_data()` call is made,
and is then cached for subsequent calls on the same reader instance.

In [ ]:
from llama_index.readers.salesforce_npsp import SalesforceNPSPReader

# --- Minimal initialization (credentials from env vars) ---
reader = SalesforceNPSPReader()
print("Reader initialized (minimal config).")
print(f"  domain              : {reader.domain}")
print(f"  include_opportunities: {reader.include_opportunities}")
print(f"  affinity_score_fn   : {reader.affinity_score_fn}")

In [ ]:
# --- Full initialization: all options explicitly set ---
# Credentials can also be passed directly instead of using env vars.
# (Here we still read from env vars to avoid hardcoding secrets.)
reader_full = SalesforceNPSPReader(
    username=os.getenv("SF_USERNAME"),
    password=os.getenv("SF_PASSWORD"),
    security_token=os.getenv("SF_TOKEN"),
    domain="login",             # "test" for sandbox orgs
    include_opportunities=True, # fetch gift history per donor
    affinity_score_fn=None,     # we will add a scorer in a later step
)
print("Reader initialized (full config, credentials passed explicitly).")

## Step 3 — Load donor documents: basic usage

`load_data()` connects to Salesforce, queries Contact records and (optionally)
their linked Opportunity gift history, and returns one `Document` per donor.

```
load_data(
    contact_ids : Optional[List[str]] = None,
    soql_filter : str = "npo02__TotalOppAmount__c > 0",
    limit       : int = 500,
) -> List[Document]
```

**Default call** — loads up to 500 contacts with any recorded lifetime giving, ordered by total
giving descending:

In [ ]:
# Default: all donors with at least one recorded gift, capped at 500.
docs = reader.load_data()

print(f"Loaded {len(docs)} donor documents.")
print(f"Return type: {type(docs)}")
print(f"Element type: {type(docs[0]) if docs else 'n/a'}")

## Step 4 — Explore the document structure

Each `Document` has two parts:
- **`text`** — a plain-text donor profile with a giving summary and gift history, ready for embedding
- **`metadata`** — a structured dictionary of numeric and string fields for filtering and display

### 4a. Document text

In [ ]:
if not docs:
    print("No documents loaded. Check credentials or SOQL filter.")
else:
    sample = docs[0]
    print("=" * 60)
    print("DOCUMENT TEXT")
    print("=" * 60)
    print(sample.text)

### 4b. Document metadata

The metadata dictionary contains key fields for programmatic filtering and analytics:

| Key | Type | Source field |
|---|---|---|
| `donor_id` | `str` | `Contact.Id` |
| `donor_name` | `str` | `Contact.FirstName + LastName` |
| `email` | `str` | `Contact.Email` |
| `affiliation` | `str` | `npsp__Primary_Affiliation__r.Name` |
| `total_gift_amount` | `float` | `npo02__TotalOppAmount__c` |
| `gift_count` | `int` | `npo02__NumberOfClosedOpps__c` |
| `average_gift_amount` | `float` | `npo02__AverageAmount__c` |
| `largest_gift_amount` | `float` | `npo02__LargestAmount__c` |
| `last_gift_date` | `str` | `npo02__LastCloseDate__c` |
| `first_gift_date` | `str` | `npo02__FirstCloseDate__c` |
| `last_activity_date` | `str` | `Contact.LastActivityDate` |
| `soft_credit_total` | `float` | `npsp__Soft_Credit_Total__c` |
| `planned_giving_count` | `int` | `npsp__Planned_Giving_Count__c` |
| `source` | `str` | Always `"salesforce_npsp"` |
| `affinity_score` | `float \| None` | Set only when `affinity_score_fn` is provided |

In [ ]:
if docs:
    sample = docs[0]
    print("=" * 60)
    print("DOCUMENT METADATA")
    print("=" * 60)
    for key, value in sample.metadata.items():
        print(f"  {key:<25} {value}")

## Step 5 — SOQL-filtered loads

The `soql_filter` parameter accepts any valid SOQL `WHERE` clause expression
applied to the `Contact` object. This lets you target specific donor segments
without pulling your entire database.

### 5a. Major donors — lifetime giving above \$10,000

In [ ]:
docs_major = reader.load_data(
    soql_filter="npo02__TotalOppAmount__c > 10000",
    limit=100,
)

print(f"Major donors (> $10k lifetime giving): {len(docs_major)} documents")
if docs_major:
    amounts = [d.metadata["total_gift_amount"] for d in docs_major]
    print(f"  Min total giving in this set: ${min(amounts):,.0f}")
    print(f"  Max total giving in this set: ${max(amounts):,.0f}")

### 5b. Lapsed donors — no gift in the past 365 days

`LAST_N_DAYS:365` is a Salesforce SOQL date literal that resolves to a rolling window
relative to today — no hard-coded date strings required.

In [ ]:
docs_lapsed = reader.load_data(
    soql_filter=(
        "npo02__TotalOppAmount__c > 0 "
        "AND npo02__LastCloseDate__c < LAST_N_DAYS:365"
    ),
    limit=200,
)

print(f"Lapsed donors (no gift in 12+ months): {len(docs_lapsed)} documents")

### 5c. High-frequency donors — 10 or more gifts on record

In [ ]:
docs_frequent = reader.load_data(
    soql_filter="npo02__NumberOfClosedOpps__c >= 10",
    limit=100,
)

print(f"High-frequency donors (10+ gifts): {len(docs_frequent)} documents")
if docs_frequent:
    counts = [d.metadata["gift_count"] for d in docs_frequent]
    print(f"  Max gifts in this set: {max(counts)}")

## Step 6 — Load specific donors by Contact ID

When you already know which donors you want — for example, from a wealth-screening
or a portfolio assignment — pass their Salesforce Contact IDs directly.
When `contact_ids` is provided, `soql_filter` and `limit` are ignored.

In [ ]:
# Replace these placeholder IDs with real Salesforce Contact IDs from your org.
# 18-character IDs starting with '003' are Contact records.
target_ids = [
    "003Dn000001xxxxxxxxx",  # replace with actual Contact ID
    "003Dn000001yyyyyyyyy",  # replace with actual Contact ID
]

docs_by_id = reader.load_data(contact_ids=target_ids)

print(f"Requested: {len(target_ids)} IDs | Loaded: {len(docs_by_id)} documents")
for doc in docs_by_id:
    print(f"  {doc.metadata['donor_id']} — {doc.metadata['donor_name']}")

## Step 7 — Affinity scoring

The `affinity_score_fn` parameter accepts any callable with the signature
`(metadata: dict) -> float`. It is called once per document during `load_data()`
and the returned float is stored as `metadata["affinity_score"]`.

This lets you inject a propensity-to-give score — from a heuristic, a trained ML model,
or an external API — directly into the document at ingestion time, so it is always
available to your query engine alongside the rest of the donor data.

The scorer below is a simple rule-based heuristic. In production you would replace
the body with a call to a trained model.

In [ ]:
def affinity_scorer(metadata: dict) -> float:
    """Rule-based propensity score derived from lifetime giving and gift frequency.

    Scores are in the range [0, 100]. Replace with a trained model in production.
    The function receives the full metadata dict for the donor and must return a float.
    """
    total = metadata.get("total_gift_amount", 0) or 0
    count = metadata.get("gift_count", 0) or 0
    # Base of 50; +1 pt per $5k total; +2 pts per gift; capped at 100
    return round(min(50 + (total / 5_000) + count * 2, 100), 1)


reader_scored = SalesforceNPSPReader(
    domain="login",
    include_opportunities=True,
    affinity_score_fn=affinity_scorer,
)

docs_scored = reader_scored.load_data(
    soql_filter="npo02__TotalOppAmount__c > 0",
    limit=200,
)

print(f"Loaded {len(docs_scored)} scored documents.")
print()
print("Top 5 donors by affinity score:")
top5 = sorted(docs_scored, key=lambda d: d.metadata.get("affinity_score") or 0, reverse=True)[:5]
for doc in top5:
    m = doc.metadata
    print(
        f"  {m['donor_name']:<30} "
        f"score={m.get('affinity_score', 'n/a'):>5}  "
        f"total=${m['total_gift_amount']:>10,.0f}  "
        f"gifts={m['gift_count']}"
    )

## Step 8 — Portfolio analytics

Because every document carries a rich metadata dictionary, you can compute
portfolio-level statistics directly from the loaded documents — no extra
database query required.

In [ ]:
# Use the scored docs loaded above; fall back to the basic docs list.
portfolio = docs_scored or docs

total_donors   = len(portfolio)
total_raised   = sum(d.metadata["total_gift_amount"] for d in portfolio)
total_gifts    = sum(d.metadata["gift_count"] for d in portfolio)
avg_gift_size  = (total_raised / total_gifts) if total_gifts else 0
planned_giving = sum(d.metadata["planned_giving_count"] for d in portfolio)
soft_credits   = sum(d.metadata["soft_credit_total"] for d in portfolio)
with_affinity  = [d for d in portfolio if d.metadata.get("affinity_score") is not None]

print("=" * 50)
print("PORTFOLIO OVERVIEW")
print("=" * 50)
print(f"  Total donors loaded     : {total_donors:>8,}")
print(f"  Total lifetime giving   : ${total_raised:>13,.0f}")
print(f"  Total gifts on record   : {total_gifts:>8,}")
print(f"  Average gift size       : ${avg_gift_size:>13,.0f}")
print(f"  Total soft credits      : ${soft_credits:>13,.0f}")
print(f"  Planned-giving contacts : {planned_giving:>8,}")
print(f"  Donors with affinity    : {len(with_affinity):>8,}")
if with_affinity:
    avg_score = sum(d.metadata["affinity_score"] for d in with_affinity) / len(with_affinity)
    print(f"  Average affinity score  : {avg_score:>8.1f}")

## Step 9 — Pre-filter documents before indexing

You can narrow the index to a specific donor segment by filtering the Python list
before passing it to `VectorStoreIndex`. This is useful when you want separate
indexes per segment (e.g. major donors vs. lapsed donors) without making additional
Salesforce API calls.

In [ ]:
# Example: focus the index on donors with $5k+ lifetime giving.
docs_to_index = [
    d for d in portfolio
    if d.metadata["total_gift_amount"] >= 5_000
]

print(f"Full portfolio  : {len(portfolio)} donors")
print(f"Filtered subset : {len(docs_to_index)} donors (>= $5k lifetime giving)")

## Step 10 — Build the vector index

`VectorStoreIndex.from_documents()` embeds each document's text, stores the vectors
in an in-memory index (default), and keeps the metadata attached for retrieval.

> **Models:** You must have an embedding model configured (and optionally an LLM for synthesis).
> If you want to use OpenAI, install `llama-index-llms-openai` + `llama-index-embeddings-openai` and set `OPENAI_API_KEY`.
> Otherwise configure `llama_index.core.Settings.embed_model` (and `Settings.llm`) before running this cell.

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(docs_to_index)
engine = index.as_query_engine()

print(f"Index built from {len(docs_to_index)} documents. Ready to query.")

## Step 11 — Fundraising Q&A

Each query uses semantic search to retrieve the most relevant donor profiles,
then synthesizes a response with the LLM. The richer the donor documents
(gift history, affinity scores, affiliations), the more nuanced the answers.

In [ ]:
# Q1: Re-engagement prospects
# Note: this reader exposes `LastActivityDate` as `metadata["last_activity_date"]`, which is a proxy for CRM activity.
response = engine.query(
    "Which donors have given over $25,000 lifetime but have had no CRM activity in over 6 months?"
)
print("Q1: Re-engagement prospects")
print("-" * 50)
print(response)

In [ ]:
# Q2: Upgrade candidates
response = engine.query(
    "Which donors have made 5 or more gifts but whose average gift is below $2,000? "
    "These are strong candidates for an upgrade ask."
)
print("Q2: Upgrade candidates")
print("-" * 50)
print(response)

In [ ]:
# Q3: Planned giving prospects
response = engine.query(
    "Which donors are already counted as planned giving prospects "
    "and have a lifetime giving total above $50,000?"
)
print("Q3: Planned giving prospects")
print("-" * 50)
print(response)

In [ ]:
# Q4: Affiliation-based prospects
response = engine.query(
    "Which donors are affiliated with a hospital or medical institution "
    "and have given over $10,000 in total?"
)
print("Q4: Affiliation-based prospects")
print("-" * 50)
print(response)

In [ ]:
# Q5: Lapsed major donors
response = engine.query(
    "Which donors gave between $5,000 and $25,000 historically but appear to have lapsed "
    "(most recent gift more than a year ago)?"
)
print("Q5: Lapsed major donors")
print("-" * 50)
print(response)

## Step 12 — Retrieval without synthesis

Use `as_retriever()` when you want the raw ranked documents back rather than
an LLM-synthesized answer — useful for building recommendation lists or
populating a CRM view.

In [ ]:
retriever = index.as_retriever(similarity_top_k=5)

nodes = retriever.retrieve(
    "Major donors who have given more than once and have a large single gift"
)

print(f"Top {len(nodes)} retrieved donors:\n")
for i, node in enumerate(nodes, 1):
    m = node.metadata
    print(
        f"  {i}. {m['donor_name']:<30} "
        f"total=${m['total_gift_amount']:>10,.0f}  "
        f"gifts={m['gift_count']}  "
        f"largest=${m['largest_gift_amount']:>10,.0f}"
    )